# 06. Train a single-dataset Bolero model

The previous section turned **ChromiumPBMC** into the units a Bolero model trains on:
a base-resolution [32 bp parquet store](../meta_cell_adata_and_parquet/04_parquet_dataset.ipynb)
and a set of [coverage-matched pseudobulks](../meta_cell_adata_and_parquet/05_pseudobulk_and_reference.ipynb) —
each carrying a cell-state embedding, a coverage scale, and a sampling weight. This page trains
the model on that **single dataset** — the simplest way to see the whole training loop end to end.

> **Single dataset vs. the full model.** The model in the paper is trained jointly across the
> entire **Bolero-10M** corpus (dozens of datasets over several species). That **multi-dataset**
> recipe reuses everything here but adds per-dataset conditioning and a genome embedding; it is
> covered in the next two pages ([multi-dataset ATAC](07_train_multi_dataset_atac.ipynb),
> [+ gene](08_train_multi_dataset_gene.ipynb)). Start here to learn the mechanics on one dataset.

**What the model is.** Bolero is a frozen [Flashzoi](https://www.biorxiv.org/content/10.1101/2024.12.18.629121v1)/Borzoi
genomics transformer wrapped in per-layer **conditional LoRA adapters**: the adapter's low-rank
weights are *generated on the fly* by small MLPs from a cell-state embedding, so every pseudobulk
gets its own effective network. The backbone weights stay frozen — training only fits the LoRA
MLPs and the output head. Here the head is the **count / accessibility** head (softplus +
Poisson-multinomial), and `signal_model=True` adds the base-resolution signal model on top.

**How the pseudobulks condition it.** Each training example is a **pair** `(p0, p1)` drawn by the
`EnsemblePairedPseudobulker` you validated on page 05: `p1` is one coverage-matched pseudobulk (a
specific cell state), `p0` is an on-the-fly ensemble background. Their 30-dim embeddings are
concatenated (30 + 30 → **60**) into the reference→target vector every LoRA MLP reads.

**In / out.**

- **Inputs** — the 32 bp parquet store and the pseudobulk records from the previous section, plus
  a frozen Flashzoi **fold-0** base checkpoint (see below).
- **Outputs** — LoRA checkpoints and epoch state under `model/` (git-ignored).

> This notebook **runs in place** and reads the artifacts you built earlier in the series via
> relative paths (`../meta_cell_adata_and_parquet/`). It is a **quick-demo** configuration — a
> handful of tiny batches so the whole loop finishes in a few minutes. The
> [production configuration](#the-production-configuration) for a real single-dataset run is
> given below.

## Setup

In [1]:
import os
from pathlib import Path

from bolero import init
from bolero.tl.model.borzoi.train import BorzoiLoRATrainer
from bolero.tl.pseudobulk.paired_pseudobulk import EnsemblePairedPseudobulker

# The trainer logs to Weights & Biases. For this demo we disable uploads so it runs without a
# login; for a real run, remove this line (or set WANDB_MODE="online") and use your own project.
os.environ["WANDB_MODE"] = "offline"

# Ray backs the streaming data pipeline (parquet -> CSR -> pseudobulk -> one-hot). num_cpus and
# object_store_memory_gb scale with the machine; these suit a single-GPU node with the data lake.
init(num_cpus=16, object_store_memory_gb=48)

2026-07-13 11:26:54,784	INFO worker.py:1781 -- Started a local Ray instance.


## Configure the run

Everything the trainer needs is a flat config dict. We first name the paths — the two inputs from
the previous section, plus the frozen backbone — then derive the embedding widths from the
pseudobulk records so the model is sized to match the data.

In [2]:
# --- Configuration --------------------------------------------------------------
DATASET_NAME = "ChromiumPBMC"
DATA_DIR = Path("../meta_cell_adata_and_parquet")   # artifacts built in the previous section

FOLD = 0                     # chromosome cross-validation fold (hg38_splits)
TARGET_COV = 5_000_000       # must match the pseudobulk records built on page 05

# inputs from the previous section
PARQUET_DIR = DATA_DIR / f"{DATASET_NAME}-MetaCell-32bp"                                # page 04
PSEUDOBULK_RECORDS = DATA_DIR / f"pseudobulk_records_and_cond.cov{TARGET_COV}.joblib"   # page 05

# frozen Flashzoi backbone (see the note below on where this comes from)
BASE_CHECKPOINT = "/home/hanliu/data/wmb/model/Borzoi/flashzoi_checkpoints/fold_0.renamed.pt"

# run identity + where checkpoints land
SAVENAME = f"{DATASET_NAME}-atac-signal-fold{FOLD}"
WANDB_PROJECT = f"BorzoiSignal-{DATASET_NAME}"

assert PARQUET_DIR.exists() and PSEUDOBULK_RECORDS.exists(), "run the previous section first"

### The frozen Flashzoi backbone

`BASE_CHECKPOINT` points to the **Flashzoi fold-0** weights that Bolero freezes and adapts. Bolero
uses the published [Flashzoi replicate](https://www.biorxiv.org/content/10.1101/2024.12.18.629121v1)
checkpoints (four cross-validation folds, on the Hugging Face Hub) with their state-dict keys
**renamed** to match this repo's `Borzoi` module — hence the `.renamed.pt` file. Point
`BASE_CHECKPOINT` at your local copy of the renamed fold-0 checkpoint; the fold here matches
`FOLD = 0`, so the frozen backbone and the chromosome CV split agree.

In [3]:
# The paired pseudobulker (page 05) both sizes the model and, at train time, samples the (p0, p1)
# pairs. Its metacell embedding is 30-dim; the reference->target concat doubles that to 60.
pseudobulker = EnsemblePairedPseudobulker(str(PSEUDOBULK_RECORDS))

emb_input_features = pseudobulker.meta_cell_emb.shape[1] * 2   # 30 -> 60 (reference + target)
cond_emb_dim = dict(pseudobulker.cond_emb_dims)                # {} for ChromiumPBMC (no conditions)

print("emb_input_features:", emb_input_features)
print("cond_emb_dim:", cond_emb_dim)

emb_input_features: 60
cond_emb_dim: {}


### Build the trainer config

`BorzoiLoRATrainer.make_config` merges these keys onto the dataset / model / trainer defaults and
validates them. The block below is the **quick-demo** run: `max_epochs`, `warmup_steps`,
`train_batches`, and `val_batches` are cut to tiny values so the full loop finishes in a few
minutes. Everything else (batch size, learning rate, gradient accumulation, the count head, the
signal model) uses the shipped defaults — the same ones a real run uses.

In [4]:
config = {
    # run identity + outputs
    "output_dir": "model",
    "savename": SAVENAME,
    "wandb_project": WANDB_PROJECT,
    "wandb_job_type": "lora",
    "wandb_name": SAVENAME,
    # data (built in the previous section)
    "dataset_path": str(PARQUET_DIR),
    "fold_split_id": FOLD,
    "pseudobulk_records": str(PSEUDOBULK_RECORDS),
    # model
    "base_checkpoint_path": BASE_CHECKPOINT,
    "emb_input_features": emb_input_features,
    "cond_emb_dim": cond_emb_dim,
    "signal_model": True,
    # ---- quick-demo overrides: a tiny run. See "The production configuration" below. ----
    "max_epochs": 3,
    "warmup_steps": 10,
    "train_batches": 50,
    "val_batches": 10,
    # At the default "batch_size" of 4 training peaks at ~57 GB of GPU memory (fits an 80 GB
    # H100). On a smaller GPU, lower it, e.g. "batch_size": 1.
}

config = BorzoiLoRATrainer.make_config(**config)

In [5]:
# A few resolved values — the demo overrides plus the inherited defaults the real run keeps.
for k in [
    "mode", "fold_split_id", "signal_model", "emb_input_features", "cond_emb_dim",
    "lr", "batch_size", "accumulate_grad",
    "max_epochs", "warmup_steps", "train_batches", "val_batches",
]:
    print(f"{k:20s} {config[k]}")

mode                 lora
fold_split_id        0
signal_model         True
emb_input_features   60
cond_emb_dim         {}
lr                   5e-05
batch_size           4
accumulate_grad      2
max_epochs           3
warmup_steps         10
train_batches        50
val_batches          10


## Train

Constructing the trainer loads the frozen backbone, freezes everything except the output head,
swaps the Linear/Conv layers for conditional-LoRA variants, and prints the model plus its
trainable-parameter count. `trainer.train()` then runs the fit loop — building the Ray data
pipeline, training and validating per epoch, and writing checkpoints under `model/`. A
`...success.flag` file is dropped on completion, so re-running the cell is a no-op (delete the flag
to retrain).

In [6]:
trainer = BorzoiLoRATrainer(config)
trainer.train()

Using single GPU for training
Create BorzoiDataset with config:
  dataset_path: ../meta_cell_adata_and_parquet/ChromiumPBMC-MetaCell-32bp
  dna_window: 524288
  pos_resolution: 32
  use_regions: borzoi
  train_region_step_sample: True
  n_pseudobulks: 10
  paired_data: True
  paired_mode: ensemble
  reverse_complement: True
  rc_mode: random
  max_jitter: 3
  shuffle_files: True
  batch_size: 4
  min_cov: 0
  normalize_cov: True
  reduce_resolution: False
  read_parquet_kwargs: None
  deg_list: None
  gene_data_path: None
  qtl_data_path: None
  _multihead: False


wandb: Tracking run with wandb version 0.28.0


wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /large_storage/zhoulab/hanliu/pkg/liuhlab/bolero/docs/tutorials/model_training/wandb/offline-run-20260713_112657-erbg54ng
wandb: View this run in the terminal with `wandb leet`


wandb: WARNING URL not available in offline run


Setting up model from config


...

 - (Validation) 3 [24/10] Mean Loss: 1.270; Mean Track Corr: 0.845; 


 - (Validation) 3 [29/10] Mean Loss: 1.247; Mean Track Corr: 0.848; 


wandb: 
wandb: Run history:
wandb:                    train/learning_rate ▁▆████████████████████████████
wandb: train/train_corr_ChromiumPBMC.MetaCell ▁▂▂▃▃▄▄▄▄▄▆▆▇▇▇▇▇▇▇▇████▇▇▇▇▇▇
wandb:                       train/train_loss ▅▄▆▄▂▅▂▂▄▄▆▆▄▆█▅▇▇█▅▁▂▂▁▂▇▂▄▁▄
wandb:           train/train_loss_multinomial ▄▄▅▄▂▅▂▂▄▄▆▆▄▆█▅▇▇█▅▁▂▂▁▂▇▂▄▁▄
wandb:               train/train_loss_poisson ▆▆▅▆█▄▇▇▅▅▂▃▅▄▁▃▂▂▁▄█▇▆█▇▃▇▅▇▅
wandb:            train/train_total_grad_norm █▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:     val/val_corr_ChromiumPBMC.MetaCell ▁▇█
wandb:                           val/val_loss █▁▃
wandb:               val/val_loss_multinomial █▁▃
wandb:                   val/val_loss_poisson ▁█▆
wandb: 
wandb: Run summary:
wandb:              final_test_corr 0.84729
wandb:              final_test_loss 1.26526
wandb:  final_test_loss_multinomial 1.84384
wandb:      final_test_loss_poisson -1.77229
wandb:             final_valid_corr 0.86911
wandb:             final_valid_loss 0.61025
wandb: final_vali

wandb: You can sync this run to the cloud by running:
wandb: wandb sync /large_storage/zhoulab/hanliu/pkg/liuhlab/bolero/docs/tutorials/model_training/wandb/offline-run-20260713_112657-erbg54ng


wandb: Find logs at: ./wandb/offline-run-20260713_112657-erbg54ng/logs


## Inspect the model

Two objects are worth looking at: the **frozen base backbone** — the plain Flashzoi/Borzoi trunk
that Bolero loads from `BASE_CHECKPOINT` and never updates — and the **full Bolero model** the
trainer just built on top of it (the same trunk, with conditional-LoRA adapters swapped into its
Linear/Conv layers, plus the output head and the signal model). Comparing the two shows where the
**trainable** parameters actually live.

In [7]:
import torch

from bolero.tl.model.borzoi.model import Borzoi, model_summary

device = trainer.device

# The frozen backbone on its own (~177M params): DNA stem -> residual conv tower + U-net ->
# transformer -> separable up-convs -> the 1920-channel trunk. This standalone copy is not frozen
# yet (torchinfo reports 0 non-trainable), but it is exactly the architecture BASE_CHECKPOINT
# fills, and these ~177M weights are the slice Bolero holds frozen inside the full model below.
base_model = Borzoi.create_from_config(config).to(device)
print(model_summary(base_model, depth=1))

Layer (type (var_name))                                                          Param #
Borzoi (Borzoi)                                                                  --
├─ConvDna (conv_dna)                                                             31,232
├─MaxPool1d (_max_pool)                                                          --
├─SequentialwithArgs (res_tower)                                                 18,592,672
├─SequentialwithArgs (unet1)                                                     9,834,496
├─SequentialwithArgs (transformer)                                               132,243,456
├─ConvBlock (horizontal_conv0)                                                   1,970,176
├─ConvBlock (horizontal_conv1)                                                   2,363,904
├─Upsample (upsample)                                                            --
├─SequentialwithArgs (upsampling_unet1)                                          2,363,904
├─ConvBlock (separable

In [8]:
# The full model. On one GPU it is the module itself; multi-GPU training would wrap it in
# DataParallel, so unwrap defensively.
model = getattr(trainer.model, "module", trainer.model)
print(model_summary(model, depth=1))

Layer (type (var_name))                                                          Param #
BorzoiLoRA (BorzoiLoRA)                                                          81,665
├─ConvDna (conv_dna)                                                             4,606,216
├─SequentialwithArgs (res_tower)                                                 201,496,352
├─Sequential (signal_encoder)                                                    2,168,832
├─SequentialwithArgs (unet1)                                                     71,185,920
├─MaxPool1d (_max_pool)                                                          --
├─ConvBlock (horizontal_conv0)                                                   23,846,400
├─ConvBlock (horizontal_conv1)                                                   26,213,888
├─SequentialwithArgs (transformer)                                               705,167,360
├─SequentialwithArgs (upsampling_unet1)                                          26,213,888
├─C

In [9]:
# Walk the top-level components and split each into total vs. trainable parameters. The backbone
# modules carry a large frozen weight plus a small conditional-LoRA adapter (the trainable part);
# the heads are fully trainable.
def count_params(module):
    total = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return total, trainable


print(f"{'component':22s} {'class':26s} {'params':>14s} {'trainable':>13s}")
print("-" * 78)
for name, child in model.named_children():
    total, trainable = count_params(child)
    if total == 0:
        continue   # e.g. the crop layer holds no parameters
    print(f"{name:22s} {type(child).__name__:26s} {total:14,d} {trainable:13,d}")
print("-" * 78)
print(f"{'TOTAL':22s} {'':26s} {trainer.total_params:14,d} {trainer.trainable_params:13,d}")
print(f"\ntrainable fraction: {trainer.trainable_params / trainer.total_params:.2%}")

component              class                              params     trainable
------------------------------------------------------------------------------
conv_dna               ConvDna                         4,606,216     4,575,496
res_tower              SequentialwithArgs            201,496,352   182,908,256
unet1                  SequentialwithArgs             71,185,920    61,352,960
transformer            SequentialwithArgs            705,167,360   572,997,632
horizontal_conv0       ConvBlock                      23,846,400    21,877,760
horizontal_conv1       ConvBlock                      26,213,888    23,851,520
upsampling_unet1       SequentialwithArgs             26,213,888    23,851,520
separable1             ConvBlock                      22,455,898    20,091,994
upsampling_unet0       SequentialwithArgs             26,213,888    23,851,520
separable0             ConvBlock                      22,455,898    20,091,994
final_joined_convs     SequentialwithArgs           

Reading the table top to bottom follows a sequence through the model:

- **`conv_dna`** — the DNA stem: one-hot 524,288 bp in, first convolutional features out.
- **`res_tower`, `unet1`, `horizontal_conv0/1`** — the residual convolution tower and U-net that
  downsample the sequence to the 32 bp bin grid while building a multi-scale representation.
- **`transformer`** — the 8 flash-attention layers, the largest block in the network. In the base
  backbone it is ~132M params; in the full model it reports **~705M**, because the conditional-LoRA
  **generators** that produce its per-layer adapter weights are nested inside it. Its trainable
  count (~573M) is those generators; the ~132M base attention / MLP weights are the frozen remainder.
- **`separable1`, `separable0`, `upsampling_unet0/1`, `final_joined_convs`** — separable
  up-convolutions that return to the 16,384-bin output grid, ending in the shared **1920-channel**
  trunk. (`crop` carries no parameters and is skipped.)
- **`final_output_head`** — the count / accessibility head (softplus); fully trainable.
- **`signal_encoder`, `cond_emb_module`** — the signal model, present because `signal_model=True`.
  `cond_emb_module` turns the paired pseudobulk embedding (and any conditions) into the vector every
  adapter generator reads; both are fully trainable. With no conditions, ChromiumPBMC's is a
  `...NoEffect` module.

The contrast between the two summaries is the point. The **frozen** part is the ~177M-parameter base
backbone — the `Non-trainable params` line in the full summary. Everything else — the
conditional-LoRA generator MLPs nested inside each backbone module, plus the heads and the signal
model — is **trainable**, and it is *not* small: adding it takes the model from **~177M to ~1.19B**
parameters, so **~85%** of the full network is trained. Bolero's conditional LoRA is a genuine
hypernetwork that regenerates the frozen backbone's effective weights for each cell state, not a thin
low-rank patch bolted on top.

## The production configuration

The single-dataset run behind the paper is the **same config with the four demo overrides removed**
— i.e. the full number of batches per epoch, over more epochs. Concretely:

```python
config = {
    # ... identical identity / data / model keys as above ...
    "signal_model": True,
    "max_epochs": 20,        # the four quick-demo lines below are simply dropped:
    # "warmup_steps": 500,   #   trainer default  (demo used 10)
    # "train_batches": 5000, #   trainer default  (demo used 50)
    # "val_batches": 300,    #   trainer default  (demo used 10)
}
```

Leaving `warmup_steps`, `train_batches`, and `val_batches` unset lets them fall back to their
defaults (**500 / 5000 / 300**). At those settings each epoch takes **~50 min on an H100 80 GB
GPU**, so a full **20-epoch** run is roughly **~17 hours**. At the default `batch_size` of 4 the
model peaks at **~57 GB of GPU memory**, so it fits comfortably on an 80 GB H100 — lower `batch_size`
if you have less. Everything else — learning rate, the count head, the signal model — is unchanged
from the demo.

For real runs, drop the `WANDB_MODE="offline"` line from the setup cell (or set it to `"online"`)
and log to your own W&B project to track the training and validation curves.

---

You now have a trained, cell-state–conditioned Bolero model for **ChromiumPBMC** under `model/`:
the frozen Flashzoi backbone plus conditional-LoRA adapters and the accessibility head, fit on the
pseudobulks built in the previous section — the whole training loop on a single dataset.

The paper's final model scales this same recipe to the **entire Bolero-10M corpus**. The next two
pages show that multi-dataset training and what changes at scale: joint training across dozens of
datasets and several genomes with per-dataset conditioning
([multi-dataset ATAC](07_train_multi_dataset_atac.ipynb)), then adding the **gene-expression** task
on top ([+ gene](08_train_multi_dataset_gene.ipynb)).